# 1. Introdução

## 1.1 Contexto

Apresentação do contexto, do modelo formal por trás da hipótese e descrição clara da própria hipótese

Contextualização da desigualdade em SP

## 1.2 Hipótese

apresentação do **Modelo Formal**.
- **Modelo Formal:** Exemplo: $Nota = \beta_0 + \beta_1(EscolaridadePais) + \beta_2(Renda) + \epsilon$. 
- **Hipótese:** Existe uma disparidade significativa nas notas do ENEM entre diferentes zonas de SP, correlacionada com indicadores socioeconômicos locais.

# 2. Dados

## 2.1 Descrição das bases de dados

3 bases de dados + Fontes

Microdados Enem 2025
https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/enem

http://www.atlasbrasil.org.br/acervo/biblioteca

https://basedosdados.org/dataset/cbfc7253-089b-44e2-8825-755e1419efc8?table=ec5fb3d1-fa98-4ab3-8a02-4b9950048a83 

Coordenadas geográficas de SP
https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_municipais/municipio_2024/UFs/SP/SP_Municipios_2024.zip

Dicionário que explique cada feature

Dicionário para evitar nomes errados
Retirada de colunas irrelevantes
Outliers (média no agrupamento por munícipio)

Fontes:

## 2.2 Leitura e Merge das bases de dados

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# 1. Carregar os dados processados do ENEM
df_enem = pd.read_csv('data/processed/enem_sp_completo_agrupado.csv')

# 2. Carregar o diretório de municípios (para garantir os nomes e siglas corretas)
df_dicionario_sp = pd.read_csv('data/raw/br_bd_diretorios_brasil_municipio.csv.gz')
df_dicionario_sp = df_dicionario_sp[df_dicionario_sp['sigla_uf'] == 'SP'][['id_municipio', 'nome']]

# 3. Carregar os dados de IDH/Infraestrutura
df_idh = pd.read_csv('data/raw/mundo_onu_adh_municipio.csv.gz')
# Filtrar apenas o ano mais recente (2010) para não duplicar linhas
df_idh = df_idh[df_idh['ano'] == 2010]

# Primeiro: Unimos o ENEM com o diretório para ter os nomes corretos das cidades
df = df_enem.merge(df_dicionario_sp, left_on='CO_MUNICIPIO_ESC', right_on='id_municipio')

# Segundo: Unimos com os dados de IDH/Infraestrutura
df = df.merge(df_idh, on='id_municipio')

# Visualizar o resultado
print(f"Total de municípios após o merge: {len(df)}")
municipios_sp = df['nome'].unique().tolist()
municipios_sp.sort()
print(f"Municípios de SP presentes no dataset: {municipios_sp}")
print(df.columns.tolist())

## 2.3 Feature engineering

In [ ]:
# Criação da nota_final juntando as notas por área do conhecimento
df['nota_final'] = (df["NU_NOTA_CN"]+df['NU_NOTA_CH']+ df['NU_NOTA_LC'] +df['NU_NOTA_MT']+ df['NU_NOTA_REDACAO'])/5
municipios = ['Ferraz de Vasconcelos', 'Indaiatuba', 'Jundiaí', 'Itaquaquecetuba', 'São Paulo', 'Poá', 'Osasco','Guarulhos','São Bernardo' ]
for i in municipios:
    df1 = df[df['nome']== i]
    colunas = ["nome",'nota_final', "idhm"]
    print(df1[colunas])
    print("---"*30)

# Média do Brasil é 546

In [ ]:

# Seleção das colunas de interesse:
colunas = ['nome', 'nota_final', 'id_municipio', 'idhm', "idhm_e", "idhm_l", "idhm_r", "indice_frequencia_escolar", "indice_escolaridade","taxa_sem_energia_eletrica", "taxa_agua_encanada", 'taxa_atividade_15_17', 'renda_media_ocupados', 'renda_pc', 'renda_pc_pobreza_extrema', 'prop_vulner_pobreza_criancas', 'prop_pobreza', 'indice_gini', 'prop_pobreza_criancas','taxa_freq_fundamental_15_17', 'taxa_analfabetismo_15_a_17']
df_final = df[colunas].set_index(['id_municipio', 'nome'])
print(df_final.head())

## 2.4 Limpeza de dados nulos

In [ ]:
# Visualização de valores nulos
print(df_final.isna().sum())
# Qual o nome do munícipio com nota final faltando?
municipios_faltando_nota = df_final[df_final['nota_final'].isna()].index.get_level_values('nome').unique().tolist()
print(f"Municípios com nota final faltando: {municipios_faltando_nota}")
df_final.dropna(subset=['nota_final'], inplace=True)
print(df_final.isna().sum())
# Quantidade de colunas agora:
print(f"Quantidade de linhas após remover linhas com nota final faltando: {df_final.shape[0]}")

In [ ]:
# Ver se tem outliers pelo método do IQR:
Q1 = df_final['nota_final'].quantile(0.25)
Q3 = df_final['nota_final'].quantile(0.75)
IQR = Q3 - Q1
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR
outliers = df_final[(df_final['nota_final'] < limite_inferior) | (df_final['nota_final'] > limite_superior)]
print(f"Municípios considerados outliers: {outliers.index.get_level_values('nome').tolist()}")
# Notas consideradas outliers:
print(outliers['nota_final'].tolist())

In [ ]:
# Winsorização(tratamento que restringe os outliers aos limites definidos anteriormente):
df_final['nota_final'] = df_final['nota_final'].clip(lower=limite_inferior, upper=limite_superior)
# Comparar as notas antes e depois da winsorização nos múnicipios considerados outliers:
df_outliers = df_final.loc[outliers.index]
print(df_outliers[['nota_final']])

## 2.5 Estatística Descritiva

In [ ]:
df_final.info()

In [ ]:
df_final.describe()

In [ ]:
df_final.shape

# 3. Resultados

## 3.1 Gráficos de dispersão

### Gráficos de dispersão dos dados das 3 maiores correlações positivas com a nota final

In [ ]:
variavel_y = 'nota_final'
variavel_x = 'idhm_r'

# 1. Calcula a correlação de Pearson para essas duas variáveis
valor_pearson = df_final[variavel_x].corr(df_final[variavel_y])

# Configura o estilo visual limpo
sns.set_theme(style="whitegrid")
plt.figure(figsize=(9, 6))

# 2. Gera a dispersão com a linha de tendência e adiciona o label para a legenda
sns.regplot(data=df_final,
            x=variavel_x, 
            y=variavel_y,
            color="#4c7dcb",
            scatter_kws={'alpha':0.6, 's':60, 'label': 'Dados Observados'}, 
            line_kws={'color':'#c44e52', 'lw':2.5, 'label': 'Linha de Regressão'}) 

# 3. Adiciona a caixa de texto com o valor de Pearson no canto superior direito
# transform=ax.transAxes faz com que as coordenadas (0.05 a 0.95) sejam relativas aos eixos do gráfico
ax = plt.gca()
texto_pearson = f'Pearson $r$ = {valor_pearson:.2f}'
plt.text(0.95, 0.95, texto_pearson, 
         transform=ax.transAxes, 
         fontsize=11, 
         weight='bold',
         color='#333333',
         verticalalignment='top', 
         horizontalalignment='right',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8, edgecolor='#cccccc'))

# Títulos e rótulos
plt.title(f'Dispersão: {variavel_x} vs {variavel_y}', fontsize=14, pad=15, weight='bold')
plt.xlabel(variavel_x, fontsize=11)
plt.ylabel(variavel_y, fontsize=11)

# 4. Ativa a legenda na tela
plt.legend(loc='lower right', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
variavel_y = 'nota_final'
variavel_x = 'renda_pc'

# 1. Calcula a correlação de Pearson dinamicamente para essas duas variáveis
valor_pearson = df_final[variavel_x].corr(df_final[variavel_y])

# Configura o estilo visual limpo
sns.set_theme(style="whitegrid")
plt.figure(figsize=(9, 6))

# 2. Gera a dispersão com a linha de tendência e adiciona o label para a legenda
sns.regplot(data=df_final,
            x=variavel_x, 
            y=variavel_y,
            color="#4c72b0",               
            scatter_kws={'alpha':0.6, 's':60, 'label': 'Dados Observados'}, 
            line_kws={'color':'#c44e52', 'lw':2.5, 'label': 'Linha de Regressão'}) 

# 3. Adiciona a caixa de texto com o valor de Pearson no canto superior direito
# transform=ax.transAxes faz com que as coordenadas (0.05 a 0.95) sejam relativas aos eixos do gráfico
ax = plt.gca()
texto_pearson = f'Pearson $r$ = {valor_pearson:.2f}'
plt.text(0.95, 0.95, texto_pearson, 
         transform=ax.transAxes, 
         fontsize=11, 
         weight='bold',
         color='#333333',
         verticalalignment='top', 
         horizontalalignment='right',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8, edgecolor='#cccccc'))

# Títulos e rótulos
plt.title(f'Dispersão: {variavel_x} vs {variavel_y}', fontsize=14, pad=15, weight='bold')
plt.xlabel(variavel_x, fontsize=11)
plt.ylabel(variavel_y, fontsize=11)

# 4. Ativa a legenda na tela
plt.legend(loc='lower right', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
variavel_y = 'nota_final'
variavel_x = 'indice_escolaridade'

# 1. Calcula a correlação de Pearson dinamicamente para essas duas variáveis
valor_pearson = df_final[variavel_x].corr(df_final[variavel_y])

# Configura o estilo visual limpo
sns.set_theme(style="whitegrid")
plt.figure(figsize=(9, 6))

# 2. Gera a dispersão com a linha de tendência e adiciona o label para a legenda
sns.regplot(data=df_final,              
            x=variavel_x, 
            y=variavel_y,
            color="#4c72b0",               
            scatter_kws={'alpha':0.6, 's':60, 'label': 'Dados Observados'}, 
            line_kws={'color':'#c44e52', 'lw':2.5, 'label': 'Linha de Regressão'}) 

# 3. Adiciona a caixa de texto com o valor de Pearson no canto superior direito
# transform=ax.transAxes faz com que as coordenadas (0.05 a 0.95) sejam relativas aos eixos do gráfico
ax = plt.gca()
texto_pearson = f'Pearson $r$ = {valor_pearson:.2f}'
plt.text(0.95, 0.95, texto_pearson, 
         transform=ax.transAxes, 
         fontsize=11, 
         weight='bold',
         color='#333333',
         verticalalignment='top', 
         horizontalalignment='right',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8, edgecolor='#cccccc'))

# Títulos e rótulos
plt.title(f'Dispersão: {variavel_x} vs {variavel_y}', fontsize=14, pad=15, weight='bold')
plt.xlabel(variavel_x, fontsize=11)
plt.ylabel(variavel_y, fontsize=11)

# 4. Ativa a legenda na tela
plt.legend(loc='lower right', fontsize=10)

plt.tight_layout()
plt.show()

### Gráficos de dispersão dos dados das 3 maiores correlações negativas com a nota final

In [ ]:
variavel_y = 'nota_final'  
variavel_x = 'prop_pobreza'     

# 1. Calcula a correlação de Pearson dinamicamente para essas duas variáveis
valor_pearson = df_final[variavel_x].corr(df_final[variavel_y])

# Configura o estilo visual limpo
sns.set_theme(style="whitegrid")
plt.figure(figsize=(9, 6))

# 2. Gera a dispersão com a linha de tendência e adiciona o label para a legenda
sns.regplot(data=df_final,              
            x=variavel_x, 
            y=variavel_y,
            color="#4c72b0",               
            scatter_kws={'alpha':0.6, 's':60, 'label': 'Dados Observados'}, 
            line_kws={'color':'#c44e52', 'lw':2.5, 'label': 'Linha de Regressão'}) 

# 3. Adiciona a caixa de texto com o valor de Pearson no canto superior direito
# transform=ax.transAxes faz com que as coordenadas (0.05 a 0.95) sejam relativas aos eixos do gráfico
ax = plt.gca()
texto_pearson = f'Pearson $r$ = {valor_pearson:.2f}'
plt.text(0.95, 0.95, texto_pearson, 
         transform=ax.transAxes, 
         fontsize=11, 
         weight='bold',
         color='#333333',
         verticalalignment='top', 
         horizontalalignment='right',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8, edgecolor='#cccccc'))

# Títulos e rótulos
plt.title(f'Dispersão: {variavel_x} vs {variavel_y}', fontsize=14, pad=15, weight='bold')
plt.xlabel(variavel_x, fontsize=11)
plt.ylabel(variavel_y, fontsize=11)

# 4. Ativa a legenda na tela
plt.legend(loc='lower right', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
variavel_y = 'nota_final'
variavel_x = 'prop_pobreza_criancas'    

# 1. Calcula a correlação de Pearson dinamicamente para essas duas variáveis
valor_pearson = df_final[variavel_x].corr(df_final[variavel_y])

# Configura o estilo visual limpo
sns.set_theme(style="whitegrid")
plt.figure(figsize=(9, 6))

# 2. Gera a dispersão com a linha de tendência e adiciona o label para a legenda
sns.regplot(data=df_final,              
            x=variavel_x, 
            y=variavel_y,
            color="#4c72b0",               
            scatter_kws={'alpha':0.6, 's':60, 'label': 'Dados Observados'}, 
            line_kws={'color':'#c44e52', 'lw':2.5, 'label': 'Linha de Regressão'}) 

# 3. Adiciona a caixa de texto com o valor de Pearson no canto superior direito
# transform=ax.transAxes faz com que as coordenadas (0.05 a 0.95) sejam relativas aos eixos do gráfico
ax = plt.gca()
texto_pearson = f'Pearson $r$ = {valor_pearson:.2f}'
plt.text(0.95, 0.95, texto_pearson, 
         transform=ax.transAxes, 
         fontsize=11, 
         weight='bold',
         color='#333333',
         verticalalignment='top', 
         horizontalalignment='right',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8, edgecolor='#cccccc'))

# Títulos e rótulos
plt.title(f'Dispersão: {variavel_x} vs {variavel_y}', fontsize=14, pad=15, weight='bold')
plt.xlabel(variavel_x, fontsize=11)
plt.ylabel(variavel_y, fontsize=11)

# 4. Ativa a legenda na tela
plt.legend(loc='lower right', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
variavel_y = 'nota_final'
variavel_x = 'prop_vulner_pobreza_criancas'

# 1. Calcula a correlação de Pearson dinamicamente para essas duas variáveis
valor_pearson = df_final[variavel_x].corr(df_final[variavel_y])

# Configura o estilo visual limpo
sns.set_theme(style="whitegrid")
plt.figure(figsize=(9, 6))

# 2. Gera a dispersão com a linha de tendência e adiciona o label para a legenda
sns.regplot(data=df_final,              
            x=variavel_x, 
            y=variavel_y,
            color="#4c72b0",               
            scatter_kws={'alpha':0.6, 's':60, 'label': 'Dados Observados'}, 
            line_kws={'color':'#c44e52', 'lw':2.5, 'label': 'Linha de Regressão'}) 

# 3. Adiciona a caixa de texto com o valor de Pearson no canto superior direito
# transform=ax.transAxes faz com que as coordenadas (0.05 a 0.95) sejam relativas aos eixos do gráfico
ax = plt.gca()
texto_pearson = f'Pearson $r$ = {valor_pearson:.2f}'
plt.text(0.95, 0.95, texto_pearson, 
         transform=ax.transAxes, 
         fontsize=11, 
         weight='bold',
         color='#333333',
         verticalalignment='top', 
         horizontalalignment='right',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8, edgecolor='#cccccc'))

# Títulos e rótulos
plt.title(f'Dispersão: {variavel_x} vs {variavel_y}', fontsize=14, pad=15, weight='bold')
plt.xlabel(variavel_x, fontsize=11)
plt.ylabel(variavel_y, fontsize=11)

# 4. Ativa a legenda na tela
plt.legend(loc='lower right', fontsize=10)

plt.tight_layout()
plt.show()

### Gráficos de dispersão dos dados das menores correlações com a nota final (positivo ou negativo)

In [ ]:
variavel_y = 'nota_final'
variavel_x = 'indice_frequencia_escolar'
# 1. Calcula a correlação de Pearson dinamicamente para essas duas variáveis
valor_pearson = df_final[variavel_x].corr(df_final[variavel_y])

# Configura o estilo visual limpo
sns.set_theme(style="whitegrid")
plt.figure(figsize=(9, 6))

# 2. Gera a dispersão com a linha de tendência e adiciona o label para a legenda
sns.regplot(data=df_final,              
            x=variavel_x, 
            y=variavel_y,
            color="#4c72b0",               
            scatter_kws={'alpha':0.6, 's':60, 'label': 'Dados Observados'}, 
            line_kws={'color':'#c44e52', 'lw':2.5, 'label': 'Linha de Regressão'}) 

# 3. Adiciona a caixa de texto com o valor de Pearson no canto superior direito
# transform=ax.transAxes faz com que as coordenadas (0.05 a 0.95) sejam relativas aos eixos do gráfico
ax = plt.gca()
texto_pearson = f'Pearson $r$ = {valor_pearson:.2f}'
plt.text(0.95, 0.95, texto_pearson, 
         transform=ax.transAxes, 
         fontsize=11, 
         weight='bold',
         color='#333333',
         verticalalignment='top', 
         horizontalalignment='right',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8, edgecolor='#cccccc'))

# Títulos e rótulos
plt.title(f'Dispersão: {variavel_x} vs {variavel_y}', fontsize=14, pad=15, weight='bold')
plt.xlabel(variavel_x, fontsize=11)
plt.ylabel(variavel_y, fontsize=11)

# 4. Ativa a legenda na tela
plt.legend(loc='lower right', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
variavel_y = 'nota_final'
variavel_x = 'taxa_analfabetismo_15_a_17'

# 1. Calcula a correlação de Pearson dinamicamente para essas duas variáveis
valor_pearson = df_final[variavel_x].corr(df_final[variavel_y])

# Configura o estilo visual limpo
sns.set_theme(style="whitegrid")
plt.figure(figsize=(9, 6))

# 2. Gera a dispersão com a linha de tendência e adiciona o label para a legenda
sns.regplot(data=df_final,              
            x=variavel_x, 
            y=variavel_y,
            color="#4c72b0",               
            scatter_kws={'alpha':0.6, 's':60, 'label': 'Dados Observados'}, 
            line_kws={'color':'#c44e52', 'lw':2.5, 'label': 'Linha de Regressão'}) 

# 3. Adiciona a caixa de texto com o valor de Pearson no canto superior direito
# transform=ax.transAxes faz com que as coordenadas (0.05 a 0.95) sejam relativas aos eixos do gráfico
ax = plt.gca()
texto_pearson = f'Pearson $r$ = {valor_pearson:.2f}'
plt.text(0.95, 0.95, texto_pearson, 
         transform=ax.transAxes, 
         fontsize=11, 
         weight='bold',
         color='#333333',
         verticalalignment='top', 
         horizontalalignment='right',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8, edgecolor='#cccccc'))

# Títulos e rótulos
plt.title(f'Dispersão: {variavel_x} vs {variavel_y}', fontsize=14, pad=15, weight='bold')
plt.xlabel(variavel_x, fontsize=11)
plt.ylabel(variavel_y, fontsize=11)

# 4. Ativa a legenda na tela
plt.legend(loc='lower right', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
variavel_y = 'nota_final'
variavel_x = 'taxa_sem_energia_eletrica'

# 1. Calcula a correlação de Pearson dinamicamente para essas duas variáveis
valor_pearson = df_final[variavel_x].corr(df_final[variavel_y])

# Configura o estilo visual limpo
sns.set_theme(style="whitegrid")
plt.figure(figsize=(9, 6))

# 2. Gera a dispersão com a linha de tendência e adiciona o label para a legenda
sns.regplot(data=df_final,              
            x=variavel_x, 
            y=variavel_y,
            color="#4c72b0",               
            scatter_kws={'alpha':0.6, 's':60, 'label': 'Dados Observados'}, 
            line_kws={'color':'#c44e52', 'lw':2.5, 'label': 'Linha de Regressão'}) 

# 3. Adiciona a caixa de texto com o valor de Pearson no canto superior direito
# transform=ax.transAxes faz com que as coordenadas (0.05 a 0.95) sejam relativas aos eixos do gráfico
ax = plt.gca()
texto_pearson = f'Pearson $r$ = {valor_pearson:.2f}'
plt.text(0.95, 0.95, texto_pearson, 
         transform=ax.transAxes, 
         fontsize=11, 
         weight='bold',
         color='#333333',
         verticalalignment='top', 
         horizontalalignment='right',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8, edgecolor='#cccccc'))

# Títulos e rótulos
plt.title(f'Dispersão: {variavel_x} vs {variavel_y}', fontsize=14, pad=15, weight='bold')
plt.xlabel(variavel_x, fontsize=11)
plt.ylabel(variavel_y, fontsize=11)

# 4. Ativa a legenda na tela
plt.legend(loc='lower right', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
variavel_y = 'nota_final'
variavel_x = 'renda_pc_pobreza_extrema'

# 1. Calcula a correlação de Pearson dinamicamente para essas duas variáveis
valor_pearson = df_final[variavel_x].corr(df_final[variavel_y])

# Configura o estilo visual limpo
sns.set_theme(style="whitegrid")
plt.figure(figsize=(9, 6))

# 2. Gera a dispersão com a linha de tendência e adiciona o label para a legenda
sns.regplot(data=df_final,              
            x=variavel_x, 
            y=variavel_y,
            color="#4c72b0",               
            scatter_kws={'alpha':0.6, 's':60, 'label': 'Dados Observados'}, 
            line_kws={'color':'#c44e52', 'lw':2.5, 'label': 'Linha de Regressão'}) 

# 3. Adiciona a caixa de texto com o valor de Pearson no canto superior direito
# transform=ax.transAxes faz com que as coordenadas (0.05 a 0.95) sejam relativas aos eixos do gráfico
ax = plt.gca()
texto_pearson = f'Pearson $r$ = {valor_pearson:.2f}'
plt.text(0.95, 0.95, texto_pearson, 
         transform=ax.transAxes, 
         fontsize=11, 
         weight='bold',
         color='#333333',
         verticalalignment='top', 
         horizontalalignment='right',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8, edgecolor='#cccccc'))

# Títulos e rótulos
plt.title(f'Dispersão: {variavel_x} vs {variavel_y}', fontsize=14, pad=15, weight='bold')
plt.xlabel(variavel_x, fontsize=11)
plt.ylabel(variavel_y, fontsize=11)

# 4. Ativa a legenda na tela
plt.legend(loc='lower right', fontsize=10)

plt.tight_layout()
plt.show()

## 3.2 Matriz de correlação

In [ ]:
# Calcula a matriz de correlação completa
matriz_corr = df_final.corr()

# 1. Aumenta significativamente a área do gráfico para expandir os quadrados
# Você pode aumentar esses números (ex: 16, 14) se ainda achar pequeno
plt.figure(figsize=(14, 12)) 

# Gera o mapa de calor
ax = sns.heatmap(matriz_corr, 
                 annot=True,        
                 cmap='coolwarm',   
                 vmin=-1, vmax=1,   
                 center=0,          
                 fmt='.2f',         
                 square=True,
                 annot_kws={"size": 10}) # Controla o tamanho da fonte dos números dentro dos quadrados

# 2. Inclina as labels do eixo X em 45 graus e alinha à direita
plt.xticks(rotation=45, ha='right')

# Garante que as labels do eixo Y fiquem 100% na horizontal
plt.yticks(rotation=0)

plt.title('Matriz de Correlação Completa', pad=20, fontsize=14)

# O tight_layout evita que os textos inclinados sejam cortados nas bordas da imagem
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Calcular a correlação e recortar apenas a coluna de interesse (nota_final)
# O uso dos dois colchetes [['...']] mantém o formato de DataFrame (matriz)
correlacoes = df_final.corr()[['nota_final']]

# 2. Ordena os valores para facilitar a visualização visual (do mais positivo ao mais negativo)
correlacoes = correlacoes.sort_values(by='nota_final', ascending=False)

# 3. Configura e plota o mapa de calor
plt.figure(figsize=(8, 10)) # Ajuste a altura (10) dependendo do número de colunas
sns.heatmap(correlacoes, 
            annot=True,       # Mostra os números dentro das células
            cmap='coolwarm',  # Esquema de cores intuitivo (vermelho = +, azul = -)
            vmin=-1, vmax=1,  # Trava a escala de correlação entre -1 e 1
            fmt='.2f')        # Arredonda para duas casas decimais

plt.title('Correlação com a Média das Notas Finais', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
variavel_de_notas = 'nota_final'
# Calculamos a correlação pelos metodos de spearman e pearson para descobrirmos as variaveis que mais se correlacionam com a nota a fim de compará-las
spearman = df_final.corr(method='spearman')[variavel_de_notas].abs()
pearson = df_final.corr(method='pearson')[variavel_de_notas].abs()
# Visualização dos resultados do método spearman
print('Váriaveis que mais se relacionam com a nota pelo método Spearman')
maiores_spearman = spearman.drop(variavel_de_notas).sort_values(ascending=False).head(10)
print(maiores_spearman)
# Visualização dos resultados do método pearson
print('Váriaveis que mais se relacionam com a nota pelo método Pearson')
maiores_pearson = pearson.drop(variavel_de_notas).sort_values(ascending=False).head(10)
print(maiores_pearson)

### 3.2.1 Qual correlação usar?

In [ ]:
from scipy import stats
stat_nota, p_nota = stats.shapiro(df_final['nota_final'])
print(f'Nota final - valor-p:{p_nota:.5f}')

if p_nota > 0.05:
    print("A variável 'nota_final' tem distribuição normal, portanto, o melhor método de correlação seria o pearson ")
if p_nota < 0.05:
    print("A variável 'nota_final' não tem distribuição normal, portanto, o melhor método de correlação seria o de spearman")
#Testando a normalidade das outras variáveis
variaveis_para_testar = ['idhm_r', 'renda_pc', 'indice_escolaridade', 'prop_pobreza','prop_pobreza_criancas','indice_frequencia_escolar','taxa_analfabetismo_15_a_17',
'taxa_sem_energia_eletrica','renda_pc_pobreza_extrema']

print(f"--- ANÁLISE DE NORMALIDADE (p-valor da Nota Final: {p_nota:.5f}) ---\n")

# 3. O 'for' vai pegar uma variável por vez da lista acima e rodar o bloco abaixo
for var in variaveis_para_testar:
    # Roda o teste de Shapiro para a variável atual do loop
    _, p_var = stats.shapiro(df_final[var])
    
    print(f"Variável: '{var}' | valor-p: {p_var:.5f}")
    
    # Se a variável atual E a nota forem normais -> Pearson
    if p_var > 0.05 and nota_eh_normal:
        print(f"-> Ambas têm distribuição normal. Melhor método: PEARSON.")
        
    # Se a variável atual for normal, mas a nota NÃO for -> Spearman
    elif p_var > 0.05 and not nota_eh_normal:
        print(f"-> '{var}' é normal, mas 'nota_final' NÃO é. Melhor método: SPEARMAN.")
        
    # Se a variável atual NÃO for normal -> Spearman (não importa a nota)
    else:
        print(f"-> '{var}' NÃO tem distribuição normal. Melhor método: SPEARMAN.")


### Correlação positiva

In [ ]:
# Olhemos agora para as correlações de pearson com resultados positivos, tirando o ''abs()'' e colocando-as em ordem descrescente
pearson = df_final.corr(method='pearson')[variavel_de_notas]
positivas_pearson = pearson.drop(variavel_de_notas).sort_values(ascending=False).head(10)
print(positivas_pearson)

### Correlação negativa

In [ ]:
# Olhemos agora para as correlações de pearson com resultados negativos, tirando o ''abs()'' e colocando-as em ordem crescente
pearson = df_final.corr(method='pearson')[variavel_de_notas]
negativas_pearson = pearson.drop(variavel_de_notas).sort_values(ascending=True).head(10)
print(negativas_pearson)

### Correlação fracas

In [ ]:
# Essas são as variáveis com menor correlação, em absoluto, com relação às notas
print('Váriaveis que menos se relacionam com a nota pelo método Spearman')
menores_spearman = spearman.drop(variavel_de_notas).sort_values(ascending=True).head(10)
print(menores_spearman)
# Visualização dos resultados do método pearson
print('Váriaveis que menos se relacionam com a nota pelo método Pearson')
menores_pearson = pearson.drop(variavel_de_notas).sort_values(ascending=True).head(10)
print(menores_pearson)

## 3.3 Mapas de calor - Geopandas

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

# Carregar o mapa de municípios de SP que baixou
mapa_sp = gpd.read_file('data/shp/SP_Municipios_2024.shp')

# Garantir que o ID do município é string para o cruzamento
mapa_sp['CD_MUN'] = mapa_sp['CD_MUN'].astype(str)
df_final_reset = df_final.reset_index()
df_final_reset['id_municipio'] = df_final_reset['id_municipio'].astype(str)

# Unir os dados alfanuméricos com o mapa geométrico
mapa_final = mapa_sp.merge(df_final_reset, left_on='CD_MUN', right_on='id_municipio')
# Plotar o mapa de calor da nota final
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
mapa_final.plot(column='nota_final', cmap='YlOrRd', legend=True, ax=ax)
plt.title('Distribuição Regional das Notas no Estado de SP')
plt.axis('off')
plt.show()

In [ ]:
# Criar o mapa de calor para idh_municipio
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
mapa_final.plot(column='idhm', cmap='YlGnBu', legend=True, ax=ax)
plt.title('Distribuição Regional do IDH dos Municípios de SP')
plt.axis('off')
plt.show()


# 4. Conclusão

Considerações finais e sugestões de passos futuros para enriquecimento da análise
- evidências foram favoráveis ou contrárias à hipótese
- Sugerir aprofundamentos
